# Transform Products Data
1. filter out invalid rows i.e rows with null product_id or duplicate product_id
2. standarize the product_category_name
3. some column names are misspelled i.e product_name_lenght, product_description_lenght change them.
4. translate the product category name to english using product_category_name_translation dataset lookup file
5. write the transformed data to the silver table


In [0]:
#Imports
from pyspark.sql.functions import col,lower, trim, broadcast

In [0]:
products_df=spark.read.table("olist_catalog.bronze.products")

### Step1 - filter out invalid rows i.e rows with null product_id or duplicate product_id

In [0]:
products_valid_df=(
    products_df.filter(col("product_id").isNotNull())
        .dropDuplicates(subset=["product_id"])
)

### Step2 - standarize the product_category_name

In [0]:
products_standardized_df = products_valid_df.withColumn('product_category_name',lower(trim(col("product_category_name"))))

### Step3 - Modify the misspelled column names

In [0]:
products_standardized2_df = products_standardized_df.withColumnsRenamed({"product_name_lenght":"product_name_length","product_description_lenght":"product_description_length"})

### Step4 - translate the product category name to english using product_category_name_translation dataset lookup file

In [0]:
product_category_name_translation_df = (
    spark.read
        .format("csv")
        .option("header","True")
        .load("/Volumes/olist_catalog/landing/files/product_category_name_translation.csv")
)

In [0]:
products_joined_df = (
    products_standardized2_df.join(
        broadcast(product_category_name_translation_df),"product_category_name","left"
    )
)

In [0]:
products_final_df = (
    products_joined_df.select("product_id",col("product_category_name_english").alias("product_category_name"),"product_name_length","product_description_length","product_photos_qty","product_weight_g","product_length_cm","product_height_cm","product_width_cm","source_file","ingestion_timestamp")
)

### Step5 - write the transformed data to the silver table

In [0]:
(
    products_final_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("olist_catalog.silver.products")
)

In [0]:
%sql
select * from olist_catalog.silver.products